In [14]:
import json

# Load configuration from config.json
with open('config.json', 'r') as f:
    config = json.load(f)

print("Configuration loaded:")
print(json.dumps(config, indent=4))

Configuration loaded:
{
    "lat": 51.22493229791279,
    "lon": 6.774635941560265,
    "kwp": 30.0,
    "tilt": 35,
    "azimuth": 0,
    "yield_factor": 0.8
}


# Configuration

Before running the code, edit the `config.json` file in the project root to specify the following attributes:

- `lat`: Latitude of the solar panel location (e.g., 51.2 for London)
- `lon`: Longitude of the solar panel location (e.g., -0.1)
- `kwp`: System peak power in kilowatts-peak (e.g., 10.0)
- `tilt`: Panel tilt angle in degrees (e.g., 35)
- `azimuth`: Panel orientation in degrees (0=South, -90=East, 90=West)
- `yield_factor`: Efficiency factor accounting for losses (e.g., 0.80)

The notebook will load these values automatically. The API forecasts use 15-minute intervals for improved granularity.

In [15]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta

def calculate_solar_forecast(lat, lon, kwp, tilt, azimuth, yield_factor=0.75):
    _ = lon
    # 1. Create a 15-minute time range for the next 24 hours
    start_time = datetime.now().replace(minute=0, second=0, microsecond=0)
    times = [start_time + timedelta(minutes=15*i) for i in range(96)]
    
    # 2. Basic Constants
    solar_constant = 1000  # Standard Test Condition (W/m2)
    day_of_year = datetime.now().timetuple().tm_yday
    
    # 3. Calculate Solar Position (Simplified)
    # Declination angle (approximate for the day)
    declination = 23.45 * np.sin(np.radians(360/365 * (day_of_year - 81)))
    
    results = []
    for t in times:
        # Hour angle: 15 degrees per hour from solar noon (12:00)
        hour_angle = 15 * (t.hour + t.minute/60 - 12)
        
        # Solar Zenith Angle (angle from directly overhead)
        cos_zenith = (np.sin(np.radians(lat)) * np.sin(np.radians(declination)) + 
                      np.cos(np.radians(lat)) * np.cos(np.radians(declination)) * 
                      np.cos(np.radians(hour_angle)))
        zenith = np.degrees(np.arccos(np.clip(cos_zenith, -1, 1)))
        
        # If sun is below horizon, output is 0
        if zenith > 90:
            power_output = 0
        else:
            # 4. Calculate Incident Angle on Tilted Surface
            # 0 azimuth = South, 90 = West, -90 = East
            # This formula finds how 'directly' the sun hits the panel
            cos_incidence = (np.cos(np.radians(zenith)) * np.cos(np.radians(tilt)) + 
                             np.sin(np.radians(zenith)) * np.sin(np.radians(tilt)) * 
                             np.cos(np.radians(hour_angle - azimuth)))
            
            # 5. Final Power Calculation (kW)
            # Power = Peak Power * Efficiency * (Actual Irradiance / 1000W/m2)
            # We assume a clear sky irradiance of ~1000W/m2 * cos(zenith)
            irradiance = solar_constant * max(0, cos_incidence)
            power_output = kwp * yield_factor * (irradiance / 1000)
            
        results.append({"Time": t.strftime("%H:%M"), "Power_kW": round(power_output, 3)})

    return pd.DataFrame(results)

# Use the loaded config
forecast_df = calculate_solar_forecast(**config)
print(forecast_df.head(20)) # Display first 5 hours

     Time  Power_kW
0   21:00       0.0
1   21:15       0.0
2   21:30       0.0
3   21:45       0.0
4   22:00       0.0
5   22:15       0.0
6   22:30       0.0
7   22:45       0.0
8   23:00       0.0
9   23:15       0.0
10  23:30       0.0
11  23:45       0.0
12  00:00       0.0
13  00:15       0.0
14  00:30       0.0
15  00:45       0.0
16  01:00       0.0
17  01:15       0.0
18  01:30       0.0
19  01:45       0.0


In [1]:
from datetime import date, timedelta
from example_files.solar import get_daily_solar_kwh

def show_summary(label: str, day_df):
    total_kwh = day_df["predicted_kwh"].sum()
    source = day_df["source"].iloc[0]
    print(f"{label}: {len(day_df)} rows, total={total_kwh:.3f} kWh, source={source}")
    print(day_df[["time", "predicted_kwh", "source"]].head(8).to_string(index=False))
    print()

# Default call -> tomorrow
tomorrow_forecast = get_daily_solar_kwh()
forecast = tomorrow_forecast
show_summary("Tomorrow", tomorrow_forecast)

# Explicit future forecast date
future_date = (date.today() + timedelta(days=3)).isoformat()
future_forecast = get_daily_solar_kwh(target_date=future_date)
show_summary(future_date, future_forecast)

# Explicit historical date
past_date = "2021-06-21"
historical_forecast = get_daily_solar_kwh(target_date=past_date)
show_summary(past_date, historical_forecast)

Tomorrow: 96 rows, total=48.039 kWh, source=forecast_api
                     time  predicted_kwh       source
2026-03-12 00:00:00+00:00            0.0 forecast_api
2026-03-12 00:15:00+00:00            0.0 forecast_api
2026-03-12 00:30:00+00:00            0.0 forecast_api
2026-03-12 00:45:00+00:00            0.0 forecast_api
2026-03-12 01:00:00+00:00            0.0 forecast_api
2026-03-12 01:15:00+00:00            0.0 forecast_api
2026-03-12 01:30:00+00:00            0.0 forecast_api
2026-03-12 01:45:00+00:00            0.0 forecast_api

2026-03-14: 96 rows, total=17.675 kWh, source=forecast_api
                     time  predicted_kwh       source
2026-03-14 00:00:00+00:00            0.0 forecast_api
2026-03-14 00:15:00+00:00            0.0 forecast_api
2026-03-14 00:30:00+00:00            0.0 forecast_api
2026-03-14 00:45:00+00:00            0.0 forecast_api
2026-03-14 01:00:00+00:00            0.0 forecast_api
2026-03-14 01:15:00+00:00            0.0 forecast_api
2026-03-14 01:30:00

# Doublecheck with real data

In [1]:
from pathlib import Path
import re
import pandas as pd

pattern = re.compile(
    r"^\s*\d+\s+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s+([-+]?\d*\.?\d+)\s+([-+]?\d*\.?\d+)\s*$"
    )

def load_solar_txt(file_path: Path) -> pd.DataFrame:
    rows = []
    for line in file_path.read_text(encoding="utf-8").splitlines():
        match = pattern.match(line)
        if match:
            rows.append(match.groups())

    df = pd.DataFrame(rows, columns=["time", "predicted_kw", "temperature_2m"])
    df["time"] = pd.to_datetime(df["time"])
    for col in ["predicted_kw", "temperature_2m"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["source"] = file_path.stem.split()[0]
    return df

txt_files = sorted(Path("data/validate").glob("*.txt"))
df_all = pd.concat([load_solar_txt(fp) for fp in txt_files], ignore_index=True)

df_all.head()

,time,predicted_kw,temperature_2m,source
0,2026-03-09 00:00:00,0.0,9.6,sun1_kw
1,2026-03-09 00:15:00,0.0,9.3,sun1_kw
2,2026-03-09 00:30:00,0.0,9.0,sun1_kw
3,2026-03-09 00:45:00,0.0,8.7,sun1_kw
4,2026-03-09 01:00:00,0.0,8.5,sun1_kw


In [18]:
# 15-minute values: kWh per row = kW * 0.25 hours
df_all["energy_kwh"] = df_all["predicted_kw"] * 0.25

energy_by_source = df_all.groupby("source", as_index=False)["energy_kwh"].sum()
total_kwh = energy_by_source["energy_kwh"].sum()

print(energy_by_source)
print(f"Total energy (all txt files): {total_kwh:.3f} kWh")

    source  energy_kwh
0  sun1_kw   16.281972
1  sun2_kw    5.913287
Total energy (all txt files): 22.195 kWh
